In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import IntegerType

BRONZE_TABLE = "medical_catalog.bronze.medical_transcriptions"
SILVER_TABLE = "medical_catalog.silver.medical_transcriptions"

df_bronze = spark.table(BRONZE_TABLE)
print("Bronze rows:", df_bronze.count())

In [0]:
df_silver = df_bronze \
    .filter(col("transcription").isNotNull()) \
    .filter(trim(col("transcription")) != "") \
    .filter(col("no").isNotNull()) \
    .filter(trim(col("no")) != "") \
    .dropDuplicates(["no"]) \
    .withColumn("medical_specialty", trim(initcap(col("medical_specialty")))) \
    .withColumn("sample_name",       trim(col("sample_name"))) \
    .withColumn("description",       trim(col("description"))) \
    .withColumn("transcription",     trim(col("transcription"))) \
    .withColumn("keywords",          trim(col("keywords")))

print("Silver rows after cleaning:", df_silver.count())

In [0]:
df_silver = df_silver \
    .withColumn("keyword_array",
        split(col("keywords"), r",\s*")) \
    .withColumn("keyword_count",
        size(col("keyword_array"))) \
    .withColumn("transcription_word_count",
        size(split(trim(col("transcription")), r"\s+"))) \
    .withColumn("transcription_char_length",
        length(col("transcription"))) \
    .withColumn("complexity_bucket",
        when(col("transcription_word_count") < 100,  "Short")
        .when(col("transcription_word_count") < 300, "Medium")
        .when(col("transcription_word_count") < 700, "Long")
        .otherwise("Very Long")) \
    .withColumn("has_keywords",
        when(
            col("keywords").isNull() | (trim(col("keywords")) == ""), lit(False)
        ).otherwise(lit(True))) \
    .withColumn("processed_at", current_timestamp())

display(df_silver.limit(5))

In [0]:
print("Final Silver rows:", df_silver.count())

# Check nulls are gone
null_counts = df_silver.select([
    count(when(isnull(c) | (col(c) == ""), c)).alias(c)
    for c in ["no", "description", "medical_specialty", "transcription"]
])
display(null_counts)

In [0]:
df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(SILVER_TABLE)

print("Silver table written!")
print("Row count:", spark.table(SILVER_TABLE).count())